# Step 7b: Proper Evaluation Metrics with Temporal Split 

## Evaluation Protocol
**Temporal Split:** 80/20 - first 80% of each user's sequence for training, remaining 20% for testing

Step 7b:
1. **ALS Normalization**: Fixed using min-max scaling from training data
2. **Readiness**: Fixed - returns 0.0 for courses without concepts
3. **NDCG**: Fixed formula for proper IDCG calculation

In [2]:
import json
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Set
import csv
import os

OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Set random seed for reproducibility
np.random.seed(42)


print("Step 7b: Proper Evaluation Metrics (FIXED)")

Step 7b: Proper Evaluation Metrics (FIXED)


In [3]:
# =============================================================================
# LOAD DATA
# =============================================================================

# Load user sequences
with open(f"{OUTPUT_PATH}/user_sequences.json", "r", encoding='utf-8') as f:
    user_sequences = json.load(f)
print(f"Loaded {len(user_sequences)} user sequences")

# Load course concepts
with open(f"{OUTPUT_PATH}/course_concepts.json", "r", encoding='utf-8') as f:
    course_concepts = json.load(f)
print(f"Loaded {len(course_concepts)} course concepts")

# Load ALS embeddings
with open(f"{OUTPUT_PATH}/als_embeddings.pkl", "rb") as f:
    als_data = pickle.load(f)
    user_embeddings = als_data["user_embeddings"]
    course_embeddings = als_data["course_embeddings"]
print(f"Loaded ALS embeddings: {len(user_embeddings)} users, {len(course_embeddings)} courses")

# Load association rules
try:
    with open(f"{OUTPUT_PATH}/association_rules.pkl", "rb") as f:
        association_rules = pickle.load(f)
except FileNotFoundError:
    with open(f"{OUTPUT_PATH}/association_rules.json", "r", encoding='utf-8') as f:
        association_rules = json.load(f)
print(f"Loaded {len(association_rules)} association rules")

# Load course prerequisites
with open(f"{OUTPUT_PATH}/course_prerequisites.pkl", "rb") as f:
    course_prereqs = pickle.load(f)
print(f"Loaded {len(course_prereqs)} course prerequisites")

# Load R-scores from CSV
R_SCORE_DICT = {}
r_matrix_path = f"{OUTPUT_PATH}/R_matrix/part-00000-85a2da65-b560-4368-8104-ac027a7cdc1f-c000.csv"
if os.path.exists(r_matrix_path):
    with open(r_matrix_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (row["user_id"], row["course_id"])
            R_SCORE_DICT[key] = float(row["R_score"])
    print(f"Loaded {len(R_SCORE_DICT)} R-score entries")
else:
    print(f"WARNING: R_matrix not found at {r_matrix_path}")

Loaded 11385 user sequences
Loaded 887 course concepts
Loaded ALS embeddings: 11385 users, 744 courses
Loaded 327 association rules
Loaded 179 course prerequisites
Loaded 18370 R-score entries


In [4]:
# =============================================================================
# COMPUTE ALS NORMALIZATION CONSTANTS (FIXED)
# =============================================================================

print("\nComputing ALS normalization constants...")
als_scores_sample = []
for user_id in list(user_embeddings.keys())[:1000]:
    for course_id in list(course_embeddings.keys())[:100]:
        score = np.dot(np.array(user_embeddings[user_id]), np.array(course_embeddings[course_id]))
        als_scores_sample.append(score)

ALS_MIN = np.min(als_scores_sample)
ALS_MAX = np.max(als_scores_sample)
print(f"ALS score range: [{ALS_MIN:.4f}, {ALS_MAX:.4f}]")

def normalize_als_score(raw_score: float) -> float:
    """Normalize ALS score to [0, 1] using min-max scaling (FIXED)"""
    if ALS_MAX == ALS_MIN:
        return 0.5
    normalized = (raw_score - ALS_MIN) / (ALS_MAX - ALS_MIN)
    return np.clip(normalized, 0, 1)


Computing ALS normalization constants...
ALS score range: [0.0236, 4.0241]


In [5]:
# =============================================================================
# TEMPORAL TRAIN/TEST SPLIT
# =============================================================================

def temporal_split(user_sequences: Dict, train_ratio: float = 0.8) -> Tuple[Dict, Dict]:
    train_data = {}
    test_data = {}
    
    for user in user_sequences:
        user_id = user["user_id"]
        sequence = user["sequence"]
        if len(sequence) < 2:
            continue
        
        split_idx = max(1, int(len(sequence) * train_ratio))
        train_courses = [item[1] for item in sequence[:split_idx]]
        test_courses = [item[1] for item in sequence[split_idx:]]
        
        if train_courses and test_courses:
            train_data[user_id] = train_courses
            test_data[user_id] = test_courses
    
    return train_data, test_data

train_data, test_data = temporal_split(user_sequences, train_ratio=0.8)

print(f"\nTemporal Split (80/20):")
print(f"  Train users: {len(train_data):,}")
print(f"  Test users: {len(test_data):,}")
print(f"  Train interactions: {sum(len(c) for c in train_data.values()):,}")
print(f"  Test interactions: {sum(len(c) for c in test_data.values()):,}")


Temporal Split (80/20):
  Train users: 11,385
  Test users: 11,385
  Train interactions: 52,352
  Test interactions: 19,127


In [6]:
# =============================================================================
# HELPER FUNCTIONS (FIXED)
# =============================================================================

def calculate_als_score(user_id: str, course_id: str) -> float:
    """Calculate raw ALS score as dot product of embeddings"""
    if user_id not in user_embeddings or course_id not in course_embeddings:
        return 0.0
    user_emb = np.array(user_embeddings[user_id])
    course_emb = np.array(course_embeddings[course_id])
    return float(np.dot(user_emb, course_emb))

def calculate_als_score_normalized(user_id: str, course_id: str) -> float:
    """Calculate normalized ALS score in [0, 1] (FIXED)"""
    raw_score = calculate_als_score(user_id, course_id)
    return normalize_als_score(raw_score)

def find_matching_rules(user_courses: List[str], target_course: str) -> List[Dict]:
    """Find association rules where antecedent matches user history"""
    user_course_set = set(user_courses)
    matching = []
    for rule in association_rules:
        antecedent = set(rule["antecedent"])
        consequent = rule["consequent"]
        if antecedent.issubset(user_course_set) and target_course in consequent:
            matching.append(rule)
    matching.sort(key=lambda x: x["confidence"], reverse=True)
    return matching

def calculate_readiness(user_id: str, target_course: str, user_train_courses: List[str]) -> float:
    """Calculate readiness score (FIXED: returns 0.0 for courses without concepts)"""
    target_concepts = set(course_concepts.get(target_course, []))
    
    # FIXED: If course has no concepts, readiness should be 0
    if not target_concepts:
        return 0.0
    
    prereq_strength = 0.0
    for mastered_course in user_train_courses:
        if mastered_course in course_prereqs:
            if target_course in course_prereqs[mastered_course]:
                strength = course_prereqs[mastered_course][target_course].get("strength", 0)
                prereq_strength = max(prereq_strength, strength)
    
    mastered_concepts = set()
    for course_id in user_train_courses:
        mastered_concepts.update(course_concepts.get(course_id, []))
    
    covered = mastered_concepts & target_concepts
    coverage = len(covered) / len(target_concepts)
    
    return 0.5 * prereq_strength + 0.5 * coverage

def get_r_score(user_id: str, course_id: str) -> float:
    """Get R-score from pre-loaded data (FIXED: from actual data)"""
    return R_SCORE_DICT.get((user_id, course_id), 0.4)

In [7]:
# =============================================================================
# RECOMMENDATION FUNCTIONS (FIXED)
# =============================================================================

def generate_hybrid_recommendations(user_id: str, train_courses: List[str],
                                     candidate_courses: List[str], top_k: int = 10) -> List[str]:
    W_ALS = 0.35
    W_RULE = 0.25
    W_RSCORE = 0.20
    W_READINESS = 0.20
    READINESS_THRESHOLD = 0.5
    READINESS_PENALTY = 0.7
    
    scores = []
    for course_id in candidate_courses:
        if course_id in train_courses:
            continue
        
        als_score = calculate_als_score_normalized(user_id, course_id)
        matching_rules = find_matching_rules(train_courses, course_id)
        rule_confidence = matching_rules[0]["confidence"] if matching_rules else 0.0
        r_score = get_r_score(user_id, course_id)
        readiness = calculate_readiness(user_id, course_id, train_courses)
        
        final_score = (W_ALS * als_score + W_RULE * rule_confidence + 
                      W_RSCORE * r_score + W_READINESS * readiness)
        
        if readiness < READINESS_THRESHOLD:
            final_score *= READINESS_PENALTY
        
        scores.append((course_id, final_score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scores[:top_k]]

def generate_als_recommendations(user_id: str, train_courses: List[str],
                                  candidate_courses: List[str], top_k: int = 10) -> List[str]:
    scores = []
    for course_id in candidate_courses:
        if course_id in train_courses:
            continue
        als_score = calculate_als_score(user_id, course_id)
        scores.append((course_id, als_score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scores[:top_k]]

def generate_popularity_recommendations(train_courses: List[str], candidate_courses: List[str], top_k: int = 10) -> List[str]:
    course_counts = defaultdict(int)
    for user_courses in train_data.values():
        for course_id in user_courses:
            course_counts[course_id] += 1
    scores = []
    for course_id in candidate_courses:
        if course_id in train_courses:
            continue
        scores.append((course_id, course_counts.get(course_id, 0)))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scores[:top_k]]

def generate_content_based_recommendations(user_id: str, train_courses: List[str],
                                            candidate_courses: List[str], top_k: int = 10) -> List[str]:
    user_concepts = set()
    for course_id in train_courses:
        user_concepts.update(course_concepts.get(course_id, []))
    scores = []
    for course_id in candidate_courses:
        if course_id in train_courses:
            continue
        course_concepts_set = set(course_concepts.get(course_id, []))
        overlap = len(user_concepts & course_concepts_set)
        scores.append((course_id, overlap))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scores[:top_k]]

In [8]:
# =============================================================================
# EVALUATION METRICS (FIXED NDCG)
# =============================================================================

def precision_at_k(recommendations: List[str], test_courses: List[str], k: int = 10) -> float:
    if not recommendations:
        return 0.0
    test_set = set(test_courses)
    hits = sum(1 for c in recommendations[:k] if c in test_set)
    return hits / min(k, len(recommendations))

def ndcg_at_k(recommendations: List[str], test_courses: List[str], k: int = 10) -> float:
    """Calculate NDCG@K (FIXED: proper IDCG calculation)"""
    test_set = set(test_courses)
    if not test_set:
        return 0.0
    
    dcg = 0.0
    for i, course_id in enumerate(recommendations[:k]):
        if course_id in test_set:
            dcg += 1.0 / np.log2(i + 2)
    
    ideal_k = min(k, len(test_set))
    if ideal_k == 0:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_k + 1))
    return dcg / idcg if idcg > 0 else 0.0

def mrr(recommendations: List[str], test_courses: List[str]) -> float:
    test_set = set(test_courses)
    for i, course_id in enumerate(recommendations):
        if course_id in test_set:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_model(model_func, sample_users: int = 1000, k: int = 10) -> Dict:
    all_users = list(test_data.keys())[:sample_users]
    all_courses = list(course_concepts.keys())
    
    precision_scores = []
    ndcg_scores = []
    mrr_scores = []
    recommended_courses = set()
    
    for idx, user_id in enumerate(all_users):
        if (idx + 1) % 100 == 0:
            print(f"  Evaluating user {idx + 1}/{len(all_users)}...")
        
        train_courses = train_data.get(user_id, [])
        test_courses = test_data.get(user_id, [])
        
        if not train_courses or not test_courses:
            continue
        
        candidate_courses = [c for c in all_courses if c not in train_courses]
        recommendations = model_func(user_id, train_courses, candidate_courses)
        recommended_courses.update(recommendations)
        
        precision_scores.append(precision_at_k(recommendations, test_courses, k))
        ndcg_scores.append(ndcg_at_k(recommendations, test_courses, k))
        mrr_scores.append(mrr(recommendations, test_courses))
    
    catalog_coverage = len(recommended_courses) / len(all_courses)
    
    return {
        "precision_at_k": np.mean(precision_scores),
        "precision_std": np.std(precision_scores),
        "ndcg_at_k": np.mean(ndcg_scores),
        "ndcg_std": np.std(ndcg_scores),
        "mrr": np.mean(mrr_scores),
        "mrr_std": np.std(mrr_scores),
        "catalog_coverage": catalog_coverage,
        "num_users": len(precision_scores)
    }

In [9]:
# =============================================================================
# RUN EVALUATION
# =============================================================================

print("="*60)
print("EVALUATION: Temporal Split with Proper Metrics (FIXED)")
print("="*60)

# Evaluate Hybrid Model
print("\n1. Evaluating Hybrid Model...")

def hybrid_model_func(user_id, train_courses, candidate_courses):
    return generate_hybrid_recommendations(user_id, train_courses, candidate_courses, top_k=10)

hybrid_results = evaluate_model(hybrid_model_func, sample_users=1000, k=10)

print(f"\nHybrid Model Results:")
print(f"  Precision@10: {hybrid_results['precision_at_k']:.4f} (+/- {hybrid_results['precision_std']:.4f})")
print(f"  NDCG@10: {hybrid_results['ndcg_at_k']:.4f} (+/- {hybrid_results['ndcg_std']:.4f})")
print(f"  MRR: {hybrid_results['mrr']:.4f} (+/- {hybrid_results['mrr_std']:.4f})")
print(f"  Catalog Coverage: {hybrid_results['catalog_coverage']:.2%}")

EVALUATION: Temporal Split with Proper Metrics (FIXED)

1. Evaluating Hybrid Model...
  Evaluating user 100/1000...
  Evaluating user 200/1000...
  Evaluating user 300/1000...
  Evaluating user 400/1000...
  Evaluating user 500/1000...
  Evaluating user 600/1000...
  Evaluating user 700/1000...
  Evaluating user 800/1000...
  Evaluating user 900/1000...
  Evaluating user 1000/1000...

Hybrid Model Results:
  Precision@10: 0.0987 (+/- 0.0664)
  NDCG@10: 0.5813 (+/- 0.3797)
  MRR: 0.5474 (+/- 0.4039)
  Catalog Coverage: 38.11%


In [10]:
# Evaluate Pure ALS
print("\n2. Evaluating Pure ALS...")

def als_model_func(user_id, train_courses, candidate_courses):
    return generate_als_recommendations(user_id, train_courses, candidate_courses, top_k=10)

als_results = evaluate_model(als_model_func, sample_users=1000, k=10)

print(f"\nPure ALS Results:")
print(f"  Precision@10: {als_results['precision_at_k']:.4f} (+/- {als_results['precision_std']:.4f})")
print(f"  NDCG@10: {als_results['ndcg_at_k']:.4f} (+/- {als_results['ndcg_std']:.4f})")
print(f"  MRR: {als_results['mrr']:.4f} (+/- {als_results['mrr_std']:.4f})")
print(f"  Catalog Coverage: {als_results['catalog_coverage']:.2%}")


2. Evaluating Pure ALS...
  Evaluating user 100/1000...
  Evaluating user 200/1000...
  Evaluating user 300/1000...
  Evaluating user 400/1000...
  Evaluating user 500/1000...
  Evaluating user 600/1000...
  Evaluating user 700/1000...
  Evaluating user 800/1000...
  Evaluating user 900/1000...
  Evaluating user 1000/1000...

Pure ALS Results:
  Precision@10: 0.1177 (+/- 0.0600)
  NDCG@10: 0.7977 (+/- 0.3223)
  MRR: 0.7935 (+/- 0.3485)
  Catalog Coverage: 33.26%


In [11]:
# Evaluate Popularity Baseline
print("\n3. Evaluating Popularity Baseline...")

def popularity_model_func(user_id, train_courses, candidate_courses):
    return generate_popularity_recommendations(train_courses, candidate_courses, top_k=10)

popularity_results = evaluate_model(popularity_model_func, sample_users=1000, k=10)

print(f"\nPopularity Baseline Results:")
print(f"  Precision@10: {popularity_results['precision_at_k']:.4f} (+/- {popularity_results['precision_std']:.4f})")
print(f"  NDCG@10: {popularity_results['ndcg_at_k']:.4f} (+/- {popularity_results['ndcg_std']:.4f})")
print(f"  MRR: {popularity_results['mrr']:.4f} (+/- {popularity_results['mrr_std']:.4f})")
print(f"  Catalog Coverage: {popularity_results['catalog_coverage']:.2%}")


3. Evaluating Popularity Baseline...
  Evaluating user 100/1000...
  Evaluating user 200/1000...
  Evaluating user 300/1000...
  Evaluating user 400/1000...
  Evaluating user 500/1000...
  Evaluating user 600/1000...
  Evaluating user 700/1000...
  Evaluating user 800/1000...
  Evaluating user 900/1000...
  Evaluating user 1000/1000...

Popularity Baseline Results:
  Precision@10: 0.0312 (+/- 0.0532)
  NDCG@10: 0.1387 (+/- 0.2594)
  MRR: 0.1192 (+/- 0.2514)
  Catalog Coverage: 3.04%


In [12]:
# Evaluate Content-Based Baseline
print("\n4. Evaluating Content-Based Baseline...")

def content_based_model_func(user_id, train_courses, candidate_courses):
    return generate_content_based_recommendations(user_id, train_courses, candidate_courses, top_k=10)

content_results = evaluate_model(content_based_model_func, sample_users=1000, k=10)

print(f"\nContent-Based Baseline Results:")
print(f"  Precision@10: {content_results['precision_at_k']:.4f} (+/- {content_results['precision_std']:.4f})")
print(f"  NDCG@10: {content_results['ndcg_at_k']:.4f} (+/- {content_results['ndcg_std']:.4f})")
print(f"  MRR: {content_results['mrr']:.4f} (+/- {content_results['mrr_std']:.4f})")
print(f"  Catalog Coverage: {content_results['catalog_coverage']:.2%}")


4. Evaluating Content-Based Baseline...
  Evaluating user 100/1000...
  Evaluating user 200/1000...
  Evaluating user 300/1000...
  Evaluating user 400/1000...
  Evaluating user 500/1000...
  Evaluating user 600/1000...
  Evaluating user 700/1000...
  Evaluating user 800/1000...
  Evaluating user 900/1000...
  Evaluating user 1000/1000...

Content-Based Baseline Results:
  Precision@10: 0.0216 (+/- 0.0421)
  NDCG@10: 0.0926 (+/- 0.2057)
  MRR: 0.0693 (+/- 0.1828)
  Catalog Coverage: 69.56%


In [13]:
# =============================================================================
# SUMMARY TABLE
# =============================================================================

comparison_df = pd.DataFrame({
    'Model': ['Hybrid', 'Pure ALS', 'Popularity', 'Content-Based'],
    'Precision@10': [
        hybrid_results['precision_at_k'],
        als_results['precision_at_k'],
        popularity_results['precision_at_k'],
        content_results['precision_at_k']
    ],
    'NDCG@10': [
        hybrid_results['ndcg_at_k'],
        als_results['ndcg_at_k'],
        popularity_results['ndcg_at_k'],
        content_results['ndcg_at_k']
    ],
    'MRR': [
        hybrid_results['mrr'],
        als_results['mrr'],
        popularity_results['mrr'],
        content_results['mrr']
    ],
    'Coverage': [
        hybrid_results['catalog_coverage'],
        als_results['catalog_coverage'],
        popularity_results['catalog_coverage'],
        content_results['catalog_coverage']
    ]
})

print("\n" + "="*70)
print("BASELINE COMPARISON SUMMARY")
print("="*70)
print(comparison_df.to_string(index=False, float_format="%.4f"))

improvement_precision = (hybrid_results['precision_at_k'] - als_results['precision_at_k']) / als_results['precision_at_k'] * 100
improvement_ndcg = (hybrid_results['ndcg_at_k'] - als_results['ndcg_at_k']) / als_results['ndcg_at_k'] * 100

print(f"\nHybrid vs Pure ALS:")
print(f"  Precision@10: {improvement_precision:+.1f}%")
print(f"  NDCG@10: {improvement_ndcg:+.1f}%")


BASELINE COMPARISON SUMMARY
        Model  Precision@10  NDCG@10    MRR  Coverage
       Hybrid        0.0987   0.5813 0.5474    0.3811
     Pure ALS        0.1177   0.7977 0.7935    0.3326
   Popularity        0.0312   0.1387 0.1192    0.0304
Content-Based        0.0216   0.0926 0.0693    0.6956

Hybrid vs Pure ALS:
  Precision@10: -16.1%
  NDCG@10: -27.1%


In [16]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

evaluation_results = {
    "hybrid": hybrid_results,
    "als": als_results,
    "popularity": popularity_results,
    "content_based": content_results,
    "temporal_split": {
        "train_users": len(train_data),
        "test_users": len(test_data),
        "train_interactions": sum(len(c) for c in train_data.values()),
        "test_interactions": sum(len(c) for c in test_data.values())
    },
    "parameters": {
        "sample_users": 1000,
        "k": 10,
        "train_ratio": 0.8,
        "als_normalization": f"min-max [{ALS_MIN:.4f}, {ALS_MAX:.4f}]"
    },
    "fixes_applied": [
        "ALS normalization: min-max scaling to [0,1]",
        "R-score: loaded from R_matrix.csv",
        "Readiness: returns 0 for courses without concepts",
        "NDCG: fixed IDCG calculation"
    ]
}

with open(f"{OUTPUT_PATH}/evaluation_results_fixed.pkl", "wb") as f:
    pickle.dump(evaluation_results, f)

with open(f"{OUTPUT_PATH}/evaluation_results_fixed.json", "w", encoding='utf-8') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\nResults saved to:")
print(f"  - {OUTPUT_PATH}/evaluation_results_fixed.pkl")
print(f"  - {OUTPUT_PATH}/evaluation_results_fixed.json")


Results saved to:
  - C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output/evaluation_results_fixed.pkl
  - C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output/evaluation_results_fixed.json


In [15]:
# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

print(f"\nKey Findings:")
print(f"  - Hybrid model: Precision@10 = {hybrid_results['precision_at_k']:.4f}")
print(f"  - Hybrid model: NDCG@10 = {hybrid_results['ndcg_at_k']:.4f}")
print(f"  - Hybrid vs ALS: Precision {improvement_precision:+.1f}%, NDCG {improvement_ndcg:+.1f}%")
print(f"  - Catalog Coverage: {hybrid_results['catalog_coverage']:.2%}")


EVALUATION COMPLETE!

Key Findings:
  - Hybrid model: Precision@10 = 0.0987
  - Hybrid model: NDCG@10 = 0.5813
  - Hybrid vs ALS: Precision -16.1%, NDCG -27.1%
  - Catalog Coverage: 38.11%
